---
title: Get started with Qiskit Functions
description: How to do common tasks, from install and authentication to checking job status and fetching error messages
---


{/* cspell:ignore Jarman, HIVQE, Cadavid, Chandarana, Leclerc, Sachdeva, HUBO, Filippov, Downfolding, Aharonov, Mundada, Yamauchi, supersymmetric, Paterakis, Gharibyan, Jaffali, Pellow */}

In [ ]:
# This cell is hidden from users
# It gets these details programmatically so we can test this notebook
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.circuit.random import random_circuit
from qiskit_ibm_catalog import QiskitFunctionsCatalog

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy().name
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")
qesem_function = catalog.load("qedma/qesem")
circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits

# Get started with Qiskit Functions
Premium, Flex, and On-Prem (through the IBM Quantum Platform API) Plan users can get started with IBM Qiskit Functions for free, or can procure a license from one of the partners who have contributed a function to the catalog.

## Request a free trial for third-party Qiskit Functions

To request a free trial, navigate to the [Qiskit Functions Catalog](https://quantum.cloud.ibm.com/functions), and explore the details panel. Click `Request a free trial` and fill out information required by the Functions partner, including the IBM Cloud `AccessGroupId`:

1. Navigate to [IBM Cloud IAM](http://cloud.ibm.com/iam/groups).
2. Verify eligibility.
   - Switch your account in the menu bar on the header to one with the following format: `XXXXXXX - [Organization Name]`
   - Ensure the organization is the same as the one associated with your Premium account.
   - If you see "[Your Name]'s Account", you are using your _personal_ account, which is not eligible for premium access.
3. Find your access group ID.
   - Click a group name.
   - Click **Details**.
   - Copy the access group ID. It should start with `AccessGroup-`.

<span id="install-qiskit-functions-catalog-client"></span>
## Install the Qiskit Functions Catalog client

1. To start using Qiskit Functions, install the IBM Qiskit Functions Catalog client:

   ```python
   pip install qiskit-ibm-catalog
   ```
1. Retrieve your API key from the [IBM Quantum Platform dashboard](https://quantum.cloud.ibm.com/), and activate your Python virtual environment.  See the [installation instructions](/docs/guides/install-qiskit#local) if you do not already have a virtual environment set up.

   <span id="save-account"></span>**If you are working in a trusted Python environment (such as on a personal laptop or workstation),** use the `save_account()` method to save your credentials locally. ([Skip to the next step](#functions-untrusted) if you are not using a trusted environment, such as a shared or public computer, to authenticate to IBM Quantum Platform.)

   To use `save_account()`, run `python` in your shell, then enter the following:

   ```python
   from qiskit_ibm_catalog import QiskitFunctionsCatalog

   QiskitFunctionsCatalog.save_account(channel="ibm_quantum_platform", token="<your-token>", instance="<instance-crn>")
   ```

   Type `exit()`. From now on, whenever you need to authenticate to the service, you can load your credentials with the following:
   ```python
   from qiskit_ibm_catalog import QiskitFunctionsCatalog
   catalog = QiskitFunctionsCatalog()
   ```
   For example:

In [ ]:
# Load saved credentials
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

<span id="functions-untrusted"></span>**Avoid executing code on an untrusted machine or an external cloud Python environment to minimize security risks.** If you must use an untrusted environment (on, for example, a public computer), change your API key after each use by deleting it on the [IBM Cloud API keys](https://cloud.ibm.com/iam/apikeys) page to reduce risk. Learn more in the [Managing user API keys](https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui) topic. To initialize the service in this situation, use this code:

```python
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# After using the following code, delete your API key on the
# IBM Quantum Platform home dashboard
catalog = QiskitFunctionsCatalog(token="<YOUR_API_KEY>") # Use the 44-character
# API_KEY you created and saved from the IBM Quantum Platform Home dashboard
```

<Admonition type="danger" title="Protect your API key">
**Never include your key in source code, Python scripts, or notebook files.** When sharing code with others, ensure that your API key is not embedded directly within the Python script. Instead, share the script without the key and provide instructions for securely setting it up.

If you accidentally share your key with someone or include it in version control like Git, immediately revoke your key by deleting it on the [IBM Cloud API keys](https://cloud.ibm.com/iam/apikeys) page to reduce risk. Learn more in the [Managing user API keys](https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui) topic.
</Admonition>

## List the functions you can access

After you authenticate, you can list the functions from the Qiskit Functions Catalog that you have access to:

In [2]:
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

## Run enabled functions

After a catalog object has been instantiated, you can select a function by using `catalog.load("<provider/function-name>")`:

In [ ]:
qesem_function = catalog.load("qedma/qesem")

Each Qiskit Function has custom inputs, options, and outputs. Check the specific documentation pages for the function you want to run for more information. By default, all users can only run one function job at a time:

In [ ]:
from qiskit.quantum_info import SparsePauliOp

avg_magnetization = SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)

job = qesem_function.run(
    pubs=[(circuit, [avg_magnetization, observable])],
    backend_name=backend_name,  # example: "ibm_fez"
    # options = {
    #     "estimate_time_only": "empirical",
    #     "default_precision": 0.2,  # Default precision is applied to all pubs that don't have a precision specified, see API reference for more details
    #     "max_execution_time": 3600,  # You can specify a maximum QPU time in seconds, see API reference for more details
    #     "transpilation_level": "standard",  # "minimal_with_layout_opt" for minimal transpilation, see API reference for more details
    #     "parallel_execution": True,  # True for parallel execution, see API reference for more details
    # },
)
job.job_id

'7f08c9d5-471b-4da2-92e7-4f2cb94c23a8'

<Admonition type="tip">

`run()` checks your remaining capacity and backend access before it submits the job. If your instance is out of capacity, or the backend you named is not accessible, `run()` raises an error right away instead of leaving the job to fail in the queue. When capacity is low, `run()` emits a warning. Pass `suppress_low_usage_warning=True` to silence it.

</Admonition>

In [ ]:
job = qesem_function.run(
    pubs=[(circuit, [avg_magnetization, observable])],
    backend_name=backend_name,  # example: "ibm_fez"
    suppress_low_usage_warning=True,
    # options = {
    #     "estimate_time_only": "empirical",
    #     "default_precision": 0.2,  # Default precision is applied to all pubs that don't have a precision specified, see API reference for more details
    #     "max_execution_time": 3600,  # You can specify a maximum QPU time in seconds, see API reference for more details
    #     "transpilation_level": "standard",  # "minimal_with_layout_opt" for minimal transpilation, see API reference for more details
    #     "parallel_execution": True,  # True for parallel execution, see API reference for more details
    # },
)

<span id="check-job-status"></span>
## Check job status

With your Qiskit Function `job_id`, you can check the status of running jobs. This includes the following statuses:

- **`QUEUED`**: The remote program is in the Qiskit Function queue. The queue priority is based on how much you've used Qiskit Functions.
- **`INITIALIZING`**: The remote program is starting; this includes setting up the remote environment and installing dependencies.
- **`RUNNING`**: The program is running. This also includes several more detailed statuses if supported by specific functions.
   - **`RUNNING: MAPPING`**: The function is currently mapping your classical inputs to quantum inputs.
   - **`RUNNING: OPTIMIZING_FOR_HARDWARE`**: The function is optimizing for the selected QPU. This could include circuit transpilation, QPU characterization, observable backpropagation, and so forth.
   - **`RUNNING: WAITING_FOR_QPU`**: The function has submitted a job to IBM Quantum Compute Service, and is waiting in the queue.
   - **`RUNNING: EXECUTING_QPU`**: The function has an active Quantum Compute job.
   - **`RUNNING: POST_PROCESSING`**: The function is post-processing results, which can include error mitigation, mapping quantum results to classical, and so forth.
- **`DONE`**: The program is complete, and you can retrieve result data with `job.result()`.
- **`ERROR`**: The program stopped running because of a problem. Use `job.result()` to get the error message.
- **`CANCELED`**: The program was canceled by a user, the service, or the server.

In [6]:
job.status()

'QUEUED'

<span id="retrieve-results"></span>
## Retrieve results
After a program is `DONE`, you can use `job.result()` to fetch the result. This output format varies with each function, so be sure to follow the specific documentation:

In [7]:
result = job.result()
print(result)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': True, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})


You can also cancel a job at any time:

In [8]:
job.cancel()

'Job has been stopped.'

## Access the associated Quantum Compute jobs

A Qiskit Function can submit one or more Quantum Compute jobs to a QPU while it runs. To retrieve the IDs of those runtime jobs, use `job.runtime_jobs()`. You can use these IDs to fetch the runtime job objects from a `QiskitRuntimeService` instance, or to find the workloads on the IBM Quantum&reg; Platform dashboard.

In [ ]:
runtime_job_ids = job.runtime_jobs()
runtime_job_ids

If a function groups its runtime jobs into sessions or batches, use `job.runtime_sessions()` to list the session IDs. Pass one session ID to `job.runtime_jobs()` to return only the runtime jobs in that session:

In [ ]:
sessions = job.runtime_sessions()
if sessions:
    session_runtime_jobs = job.runtime_jobs(runtime_session=sessions[0])
    print(session_runtime_jobs)
else:
    print("No runtime sessions for this job.")

<Admonition type="note">
The returned list can be empty. A function reports its runtime jobs only when it submits them through the runtime service that the function receives at run time, and some functions do not submit runtime jobs directly.
</Admonition>

## View job logs

Use `job.logs()` to retrieve the log output that a function produces while it runs. Logs are useful for tracking progress and for debugging a job that ends in an `ERROR` state.

In [ ]:
print(job.logs().splitlines())

For a long-running job that produces many log lines, use `job.filtered_logs()` to return only the lines you want. Pass a regular expression to `include` to keep matching lines, or to `exclude` to drop matching lines:

In [ ]:
print(job.filtered_logs(include="iteration"))

## List previously run Qiskit Functions jobs

You can use `jobs()` to list all jobs submitted to Qiskit Functions:

In [9]:
old_jobs = catalog.jobs()
old_jobs

[<Job | f6c29f49-4d5f-4fff-aca6-2e9a115b9763>,
 <Job | 7f08c9d5-471b-4da2-92e7-4f2cb94c23a8>,
 <Job | 62fe9176-d1e5-467e-b2bd-7a3f3c7be4e5>,
 <Job | af525b2e-16b1-45a1-80bb-dbd94ce30258>,
 <Job | b95a7a57-c1ad-4958-b7ac-953e4e1ee824>,
 <Job | 7bfa33da-0f17-4e67-84b6-f556f7eeb436>,
 <Job | ca46c191-9eb9-4de6-bfa7-b60d7eb29b5e>,
 <Job | 6ac0ba93-3831-43fb-9fb9-760da2225e06>,
 <Job | f0e38071-060d-47e8-988d-9cc1f69358e3>,
 <Job | 629cf110-e490-4675-8a07-f6d298d166b0>]

To narrow the results, pass filters. Filter by function with `function`, by status with `status`, and by submission date with `created_after`. Page through results with `limit` and `offset`:

In [ ]:
recent_errors = catalog.jobs(
    function=qesem_function,
    status="ERROR",
    created_after="2024-01-01T00:00:00Z",
    limit=5,
)
recent_errors

If you already have the job ID for a certain job, you can retrieve the job with `catalog.job()`:

In [18]:
# First, get the most recent job that has been executed.
latest_job = old_jobs[0]

# We can also get that same job with `catalog.job`
job_by_id = catalog.job(latest_job.job_id)

# Verify that the job is the same using both retrieval methods.
assert job_by_id.job_id == latest_job.job_id

# Print the job_id for this job.
print(job_by_id.job_id)

f6c29f49-4d5f-4fff-aca6-2e9a115b9763


## Fetch error messages

If a program status is `ERROR`, use `job.error_message()` to fetch the error message as follows:

In [1]:
job.error_message()

qiskit.exceptions.QiskitError: 'Workflow execution failed -- https://docs.quantum.ibm.com/errors#9999'


## Next steps

<Admonition type="info" title="Recommendations">

- [Explore circuit functions](/docs/guides/algorithmiq-tem) to build new algorithms and applications, without needing to manage transpilation or error handling.
- [Explore application functions](/docs/guides/q-ctrl-optimization-solver) to solve domain-specific tasks, with classical inputs and outputs.
- See the [API reference documentation](/docs/api/functions/index) for Qiskit Functions.
- For hands-on experience, try out some [tutorials](/docs/tutorials/index#leverage-qiskit-capabilities) that demonstrate Qiskit Functions.
</Admonition>